In [ ]:
!pip install fastapi uvicorn rembg[gpu] pyngrok nest-asyncio python-multipart

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.3/53.3 MB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 81.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 519.2/519.2 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.8/161.8 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.5/61.5 MB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 439.3/439.3 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.5/42.5 MB 5.7 MB/s eta 0:00:00


In [ ]:
import numpy as np
from PIL import Image
from fastapi import FastAPI, Request
from fastapi.responses import Response, JSONResponse

!pip install fastapi uvicorn rembg[gpu] pyngrok nest-asyncio python-multipart
!pip uninstall -y -q onnxruntime onnxruntime-gpu
!pip install -q "onnxruntime-gpu[cuda,cudnn]"
# Free port 8000 & kill old tunnel
!fuser -k 8000/tcp || true
!pkill -f cloudflared || true
!pip install -q onnxruntime-gpu

# ==============================================================================
# MC THÙY TRANG - REMOTE AI CONTROLLER FOR ANTIGRAVITY
# Direct access to: /content/drive/MyDrive/MC Thùy Trang - testing/Ảnh
# ==============================================================================

!pip uninstall -y -q opencv-python opencv-contrib-python opencv-python-headless opencv-contrib-python-headless
!rm -rf /usr/local/lib/python3.13/dist-packages/cv2 /usr/local/lib/python3.13/dist-packages/opencv* /content/cv2
!pip install -q --no-cache-dir --force-reinstall --no-deps "opencv-python-headless<5"
!pip install -q fastapi uvicorn rembg[gpu] python-multipart nest_asyncio pillow numpy

from google.colab import drive
import os, io, sys, re, cv2, time, threading, subprocess, json


if not hasattr(cv2, "CascadeClassifier"):
    # A broken cv2 is cached in this kernel; only a restart clears it. Check the files on disk are good first.
    chk = subprocess.run([sys.executable, "-c", "import cv2; print(cv2.__file__, list(cv2.__path__)); cv2.CascadeClassifier"], capture_output=True, text=True, cwd="/tmp")
    ok = chk.returncode == 0
    if not ok:
        raise RuntimeError(f"cv2 is still broken on disk.\nstdout: {chk.stdout}\nstderr: {chk.stderr[-800:]}\nhost cv2: {getattr(cv2, '__file__', None)} {list(getattr(cv2, '__path__', []))}")
    print("⚠️ Stale cv2 cached in this kernel. Restarting the runtime now. Re-run this cell once it reconnects.")
    os.kill(os.getpid(), 9)
import onnxruntime as ort
ort.preload_dlls()  # load CUDA/cuDNN from the pip packages before rembg touches them
print("Providers:", ort.get_available_providers())
from rembg import remove, new_session
import nest_asyncio
import uvicorn

# 1. Mount Drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

BASE_PHOTOS = "/content/drive/MyDrive/MC Thùy Trang - testing/Ảnh chưa edit"
print(f"📁 Root Photos Path: {BASE_PHOTOS} (Exists: {os.path.exists(BASE_PHOTOS)})")

app = FastAPI(title="Antigravity Colab Remote Controller")

# ------------------------------------------------------------------------------
# MATTING: model chain + Mask Quality Score (MQS) + edge decontamination
# ------------------------------------------------------------------------------
from pymatting import estimate_foreground_ml

PROVIDERS = ["CUDAExecutionProvider", "CPUExecutionProvider"]
MODEL_CHAIN = ["birefnet-portrait", "birefnet-general", "isnet-general-use"]  # tried in order
MQS_PASS = 80          # stop at the first model scoring >= this; otherwise keep the best
_sessions = {}

def get_session(name):
    if name not in _sessions:
        _sessions[name] = new_session(name, providers=PROVIDERS)
    return _sessions[name]

session = None         # primary session: also used by /montage-drive and /exec
for _m in MODEL_CHAIN:
    try:
        session = get_session(_m)
        print("Primary matting model:", _m)
        break
    except Exception as e:
        print(f"⚠️ {_m} failed to load: {e}")
if session is None:
    session = get_session("u2net")

def _small_holes(solid, max_frac=0.002):
    """Enclosed background pockets smaller than max_frac of the image and not touching the border."""
    H, W = solid.shape
    n, lab, st, _ = cv2.connectedComponentsWithStats((~solid.astype(bool)).astype(np.uint8), connectivity=4)
    x, y, w, h, ar = st[:, 0], st[:, 1], st[:, 2], st[:, 3], st[:, 4]
    ok = (ar < max_frac * H * W) & (x > 0) & (y > 0) & (x + w < W) & (y + h < H)
    ok[0] = False
    return ok[lab]

def mask_quality(alpha):
    """MQS 0-100 for a raw uint8 alpha. Heuristic weights: calibrate them on your own photos."""
    H, W = alpha.shape
    solid = (alpha > 127).astype(np.uint8)
    area = int(solid.sum())
    if area == 0:
        return 0.0, {"empty": True}
    k = np.ones((9, 9), np.uint8)
    band = (cv2.dilate(solid, k) - cv2.erode(solid, k)) > 0
    fringe = float(((alpha > 25) & (alpha < 230))[band].mean()) if band.any() else 0.0  # grey share of the edge band
    n, lab, st, _ = cv2.connectedComponentsWithStats(solid, connectivity=8)
    areas = st[1:, cv2.CC_STAT_AREA]
    specks = float(areas[areas < 0.01 * area].sum()) / area        # tiny detached blobs
    holes = float(_small_holes(solid).sum()) / area                # pinholes inside the subject
    cov = area / (H * W)
    score = (100
             - 45 * np.clip((fringe - 0.15) / 0.45, 0, 1)
             - 25 * np.clip(specks / 0.03, 0, 1)
             - 15 * np.clip(holes / 0.01, 0, 1)
             - (15 if cov < 0.01 or cov > 0.9 else 0))
    return float(score), {"fringe": round(fringe, 3), "specks": round(specks, 4), "holes": round(holes, 4), "coverage": round(cov, 3)}

def refine_alpha(alpha, lo=0.06, hi=0.94):
    """Drop specks, fill pinholes, then smoothstep away faint grey alpha (the dirty halo)."""
    a = alpha.astype(np.float32) / 255.0
    n, lab, st, _ = cv2.connectedComponentsWithStats((a > 0.5).astype(np.uint8), connectivity=8)
    if n > 1:
        areas = st[1:, cv2.CC_STAT_AREA]
        keep = np.r_[False, areas >= 0.01 * areas.sum()]
        near = cv2.dilate(keep[lab].astype(np.uint8), np.ones((15, 15), np.uint8)) > 0
        a = np.where(near, a, 0).astype(np.float32)
    a[_small_holes(a > 0.5)] = 1.0
    t = np.clip((a - lo) / (hi - lo), 0, 1)
    return (t * t * (3 - 2 * t) * 255).astype(np.uint8)

def best_cutout(img, model=None, max_side=3000):
    """Run the model chain, keep the best-MQS mask, refine it and decontaminate edge colours."""
    rgb = img.convert("RGB")
    if max_side and max(rgb.size) > max_side:     # keeps matting fast and memory-safe; max_side=0 disables
        s = max_side / max(rgb.size)
        rgb = rgb.resize((round(rgb.width * s), round(rgb.height * s)), Image.LANCZOS)
    best, tried = None, []
    for name in ([model] if model else MODEL_CHAIN):
        try:
            raw = np.array(remove(rgb, session=get_session(name), only_mask=True))
        except Exception as e:
            tried.append({"model": name, "error": str(e)[:120]})
            continue
        score, met = mask_quality(raw)
        tried.append({"model": name, "mqs": round(score, 1), **met})
        if best is None or score > best[0]:
            best = (score, name, raw)
        if score >= MQS_PASS:
            break
    if best is None:
        raise RuntimeError(f"No matting model produced a mask: {tried}")
    score, name, raw = best
    alpha = refine_alpha(raw)
    fg = estimate_foreground_ml(np.asarray(rgb, np.float64) / 255.0, alpha.astype(np.float64) / 255.0)
    out = np.dstack([np.clip(fg * 255, 0, 255).astype(np.uint8), alpha])
    return Image.fromarray(out), {"model": name, "mqs": round(score, 1), "tried": tried}

# ------------------------------------------------------------------------------
# 1. EXPLORE: List subfolders and files in Drive
# ------------------------------------------------------------------------------
@app.get("/list")
def list_drive(folder: str = ""):
    target = os.path.join(BASE_PHOTOS, folder) if folder else BASE_PHOTOS
    if not os.path.exists(target):
        return {"error": f"Path not found: {target}"}
    items = []
    for entry in sorted(os.listdir(target)):
        full = os.path.join(target, entry)
        items.append({
            "name": entry,
            "is_dir": os.path.isdir(full),
            "size_kb": round(os.path.getsize(full) / 1024, 1) if not os.path.isdir(full) else None
        })
    return {"path": target, "items": items}

# ------------------------------------------------------------------------------
# 2. CUTOUT: Process any photo directly from Drive -> Returns transparent PNG
# ------------------------------------------------------------------------------
@app.post("/cutout-drive")
async def cutout_drive(req: Request):
    data = await req.json()
    rel_path = data.get("path") # e.g. "Event - Danone.../photo.jpg"
    full_path = os.path.join(BASE_PHOTOS, rel_path)
    if not os.path.exists(full_path):
        return JSONResponse({"error": f"File not found: {full_path}"}, status_code=404)

    img = Image.open(full_path).convert("RGBA")
    cutout_img, mqs_report = best_cutout(img, data.get("model"), data.get("max_side", 3000))

    # Auto-crop bounding box with padding
    bbox = cutout_img.getbbox()
    if bbox:
        pad = 10
        w, h = cutout_img.size
        crop_box = (max(0, bbox[0]-pad), max(0, bbox[1]-pad), min(w, bbox[2]+pad), min(h, bbox[3]+pad))
        cutout_img = cutout_img.crop(crop_box)

    buf = io.BytesIO()
    cutout_img.save(buf, format="PNG")
    return Response(content=buf.getvalue(), media_type="image/png",
                    headers={"X-Mask-QC": json.dumps(mqs_report)})

# ------------------------------------------------------------------------------
# MONTAGE HELPERS: subjects -> align -> layout -> colour -> seam -> pyramid blend
# ------------------------------------------------------------------------------
CFG = {
    "w_color": 1.0, "w_grad": 0.5, "w_edge": 0.1,        # image-difference terms
    "w_subject": 100.0, "w_head": 500.0, "w_face": 2000.0,  # protected-area penalties
    "hard_block": 1e6,    # added on every protected pixel: seam enters only if no alternative exists
    "cross_tol": 0.01,    # QC: max fraction of seam rows allowed inside protected pixels
    "levels": 4,          # Laplacian pyramid depth
}
_face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")

def analyze_subjects(bgr):
    """rembg mask + face/head penalties. Returns (subject_mask 0/1, penalty_map)."""
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    m = np.array(remove(Image.fromarray(rgb), session=session, only_mask=True))
    subj = cv2.dilate((m > 127).astype(np.float32), np.ones((15, 15), np.uint8))  # safety halo
    pen = CFG["w_subject"] * subj
    H, W = bgr.shape[:2]
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    for (x, y, w, h) in _face_cascade.detectMultiScale(gray, 1.1, 5, minSize=(60, 60)):
        x0, y0, x1, y1 = max(0, x - w // 2), max(0, y - h // 2), min(W, x + w + w // 2), min(H, y + h + h // 4)
        pen[y0:y1, x0:x1] = np.maximum(pen[y0:y1, x0:x1], CFG["w_head"])
        pen[y:y + h, x:x + w] = np.maximum(pen[y:y + h, x:x + w], CFG["w_face"])
    return subj, pen

def align_to(ref, img, ref_subj, img_subj, min_inliers=40):
    """Homography from img onto ref using background features only. None if not confident."""
    orb = cv2.ORB_create(4000)
    k1, d1 = orb.detectAndCompute(cv2.cvtColor(ref, cv2.COLOR_BGR2GRAY), ((1 - ref_subj) * 255).astype(np.uint8))
    k2, d2 = orb.detectAndCompute(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY), ((1 - img_subj) * 255).astype(np.uint8))
    if d1 is None or d2 is None:
        return None
    matches = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True).match(d2, d1)
    if len(matches) < min_inliers:
        return None
    src = np.float32([k2[m.queryIdx].pt for m in matches]).reshape(-1, 1, 2)
    dst = np.float32([k1[m.trainIdx].pt for m in matches]).reshape(-1, 1, 2)
    Hm, inl = cv2.findHomography(src, dst, cv2.RANSAC, 4.0)
    if Hm is None or inl.sum() < min_inliers or inl.mean() < 0.35: # not enough good points
        return None
    if not 0.8 < np.sqrt(abs(np.linalg.det(Hm[:2, :2]))) < 1.25:   # reject wild warps
        return None
    return Hm

def _tile(a, x0, tw):
    a = a[:, x0:x0 + tw]
    pad = tw - a.shape[1]
    return cv2.copyMakeBorder(a, 0, 0, 0, pad, cv2.BORDER_REPLICATE) if pad > 0 else a

def bg_stats(bgr, subj):
    """Lab mean/std from background pixels only (no subject halo, no skin)."""
    lab = cv2.cvtColor(bgr.astype(np.float32) / 255.0, cv2.COLOR_BGR2Lab)
    skin = cv2.inRange(cv2.cvtColor(bgr, cv2.COLOR_BGR2YCrCb), (0, 133, 77), (255, 173, 127)) > 0
    keep = (subj < 0.5) & ~skin
    if keep.mean() < 0.05:
        keep = np.ones_like(keep)
    px = lab[keep]
    return px.mean(0), px.std(0) + 1e-6

def reinhard(bgr, subj, ref_stats):
    mu_s, sd_s = bg_stats(bgr, subj)
    mu_r, sd_r = ref_stats
    lab = cv2.cvtColor(bgr.astype(np.float32) / 255.0, cv2.COLOR_BGR2Lab)
    lab = (lab - mu_s) * np.clip(sd_r / sd_s, 0.7, 1.4) + mu_r
    out = cv2.cvtColor(lab.astype(np.float32), cv2.COLOR_Lab2BGR)
    return np.clip(out * 255.0, 0, 255).astype(np.uint8)

def _grads(bgr):
    g = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY).astype(np.float32)
    return np.dstack([cv2.Sobel(g, cv2.CV_32F, 1, 0), cv2.Sobel(g, cv2.CV_32F, 0, 1)]) / 8.0

def seam_energy(A, B, pA, pB):
    la = cv2.cvtColor(A.astype(np.float32) / 255.0, cv2.COLOR_BGR2Lab)
    lb = cv2.cvtColor(B.astype(np.float32) / 255.0, cv2.COLOR_BGR2Lab)
    ga, gb = _grads(A), _grads(B)
    color = np.linalg.norm(la - lb, axis=2)
    grad = np.linalg.norm(ga - gb, axis=2)
    edge = np.maximum(np.linalg.norm(ga, axis=2), np.linalg.norm(gb, axis=2))
    pen = np.maximum(pA, pB)
    E = CFG["w_color"] * color + CFG["w_grad"] * grad + CFG["w_edge"] * edge + pen
    E = E + CFG["hard_block"] * (pen > 0)
    m = 2 ** (CFG["levels"] + 1)          # keep the seam off the strip borders (pyramid leakage)
    E[:, :m] += 1e4
    E[:, -m:] += 1e4
    return E.astype(np.float64), color

def find_seam(E):
    """Vertical min-cost path; seam[y] = first column taken from image B."""
    h, w = E.shape
    dp, back, cols = E.copy(), np.zeros((h, w), np.int8), np.arange(w)
    for y in range(1, h):
        prev = dp[y - 1]
        stack = np.stack([np.r_[np.inf, prev[:-1]], prev, np.r_[prev[1:], np.inf]])
        idx = stack.argmin(0)
        dp[y] += stack[idx, cols]
        back[y] = idx - 1
    seam = np.zeros(h, np.int32)
    seam[-1] = dp[-1].argmin()
    for y in range(h - 1, 0, -1):
        seam[y - 1] = seam[y] + back[y, seam[y]]
    return seam

def laplacian_blend(A, B, mask, levels):
    A, B = A.astype(np.float32), B.astype(np.float32)
    def gp(x):
        p = [x]
        for _ in range(levels):
            p.append(cv2.pyrDown(p[-1]))
        return p
    def lp(g):
        out = [g[i] - cv2.pyrUp(g[i + 1], dstsize=(g[i].shape[1], g[i].shape[0])) for i in range(levels)]
        return out + [g[-1]]
    la, lb, gm = lp(gp(A)), lp(gp(B)), gp(mask.astype(np.float32))
    mixed = [a * m[..., None] + b * (1 - m[..., None]) for a, b, m in zip(la, lb, gm)]
    out = mixed[-1]
    for lvl in reversed(mixed[:-1]):
        out = cv2.pyrUp(out, dstsize=(lvl.shape[1], lvl.shape[0])) + lvl
    return np.clip(out, 0, 255).astype(np.uint8)

def build_montage(frames, subjs, pens, target_w, target_h, overlap):
    n = len(frames)
    slice_w = (target_w - overlap) // n
    tw = slice_w + overlap
    tiles, t_subj, t_pen = [], [], []
    # subject-aware layout: centre each subject in the region its tile owns
    for i, (im, sj, pn) in enumerate(zip(frames, subjs, pens)):
        cols = sj.sum(0)
        cx = int((cols * np.arange(len(cols))).sum() / cols.sum()) if cols.sum() > 0 else im.shape[1] // 2
        lo = 0 if i == 0 else overlap
        hi = tw if i == n - 1 else slice_w
        x0 = int(np.clip(cx - (lo + hi) // 2, 0, max(0, im.shape[1] - tw)))
        tiles.append(_tile(im, x0, tw)); t_subj.append(_tile(sj, x0, tw)); t_pen.append(_tile(pn, x0, tw))
    # colour normalisation toward the middle tile (background-only Lab stats)
    anchor = n // 2
    a_stats = bg_stats(tiles[anchor], t_subj[anchor])
    for i in range(n):
        if i != anchor:
            tiles[i] = reinhard(tiles[i], t_subj[i], a_stats)
    # optimal seam + multi-scale blend, one join at a time
    cur, cur_pen = tiles[0], t_pen[0]
    crossings, seam_diffs = [], []
    rows = np.arange(target_h)
    for i in range(1, n):
        A, B = cur[:, -overlap:], tiles[i][:, :overlap]
        pA, pB = cur_pen[:, -overlap:], t_pen[i][:, :overlap]
        E, color = seam_energy(A, B, pA, pB)
        seam = find_seam(E)
        mask = (np.arange(overlap)[None, :] < seam[:, None]).astype(np.float32)   # 1 = take A
        blend = laplacian_blend(A, B, mask, CFG["levels"])
        pen_blend = np.where(mask > 0.5, pA, pB)
        sc = np.minimum(seam, overlap - 1)
        crossings.append(float((np.maximum(pA, pB)[rows, sc] > 0).mean()))
        seam_diffs.append(float(color[rows, sc].mean()))
        cur = np.hstack([cur[:, :-overlap], blend, tiles[i][:, overlap:]])
        cur_pen = np.hstack([cur_pen[:, :-overlap], pen_blend, t_pen[i][:, overlap:]])
    return cur, {"max_cross": max(crossings), "mean_seam_diff": float(np.mean(seam_diffs)), "overlap": overlap}

# ------------------------------------------------------------------------------
# 3. MONTAGE: Multi-band frequency blend of Drive photos -> Returns JPEG
# ------------------------------------------------------------------------------
@app.post("/montage-drive")
async def montage_drive(req: Request):
    data = await req.json()
    paths = data.get("paths", []) # list of relative paths
    target_w, target_h = data.get("size", [2400, 1350])

    loaded = []
    for p in paths:
        full = os.path.join(BASE_PHOTOS, p)
        if os.path.exists(full):
            im = cv2.imread(full)
            h, w = im.shape[:2]
            im_resized = cv2.resize(im, (int(w * (target_h / h)), target_h), interpolation=cv2.INTER_LANCZOS4)
            loaded.append(im_resized)

    if len(loaded) < 2:
        return JSONResponse({"error": "Need at least 2 valid photos"}, status_code=400)

    # 1) subject analysis (rembg mask + face/head penalties)
    subjs, pens = [], []
    for im in loaded:
        s, p = analyze_subjects(im)
        subjs.append(s); pens.append(p)

    # 2) geometric alignment to the first frame (only applied when confident)
    aligned, ref = 0, loaded[0]
    for i in range(1, len(loaded)):
        Hm = align_to(ref, loaded[i], subjs[0], subjs[i])
        if Hm is not None:
            size = (ref.shape[1], ref.shape[0])
            loaded[i] = cv2.warpPerspective(loaded[i], Hm, size, borderMode=cv2.BORDER_REPLICATE)
            subjs[i] = cv2.warpPerspective(subjs[i], Hm, size)
            pens[i] = cv2.warpPerspective(pens[i], Hm, size)
            aligned += 1

    # 3-6) layout -> colour match -> optimal seam -> Laplacian blend
    overlap = int(data.get("overlap", 260))
    current, qc = build_montage(loaded, subjs, pens, target_w, target_h, overlap)

    # 7) quality check: retry with a wider overlap if the seam cut a protected area
    if qc["max_cross"] > CFG["cross_tol"]:
        alt, alt_qc = build_montage(loaded, subjs, pens, target_w, target_h, int(overlap * 1.6))
        if alt_qc["max_cross"] < qc["max_cross"]:
            current, qc = alt, alt_qc
    qc["aligned"] = aligned

    final_montage = cv2.resize(current, (target_w, target_h), interpolation=cv2.INTER_LANCZOS4)

    # Soft cinematic vignette
    Y, X = np.ogrid[:target_h, :target_w]
    vignette = 1.0 - 0.35 * (np.sqrt((X - target_w/2)**2 + (Y - target_h/2)**2) / np.sqrt((target_w/2)**2 + (target_h/2)**2))**1.5
    vignette = cv2.merge([vignette, vignette, vignette])
    final_montage = np.clip(final_montage * vignette, 0, 255).astype(np.uint8)

    _, encoded = cv2.imencode(".jpg", final_montage, [cv2.IMWRITE_JPEG_QUALITY, 95])
    return Response(content=encoded.tobytes(), media_type="image/jpeg",
                    headers={"X-Montage-QC": json.dumps(qc)})

# ------------------------------------------------------------------------------
# 4. REMOTE CODE EXECUTION: Maintain logic in Antigravity, run on Colab GPU
# ------------------------------------------------------------------------------
@app.post("/exec")
async def execute_code(req: Request):
    data = await req.json()
    code_str = data.get("code", "")

    old_stdout = sys.stdout
    old_stderr = sys.stderr
    redirected_output = io.StringIO()
    redirected_error = io.StringIO()
    sys.stdout = redirected_output
    sys.stderr = redirected_error

    exec_globals = {
        "BASE_PHOTOS": BASE_PHOTOS,
        "cv2": cv2,
        "np": np,
        "Image": Image,
        "remove": remove,
        "session": session,
        "os": os
    }

    try:
        exec(code_str, exec_globals)
        success = True
    except Exception as e:
        import traceback
        traceback.print_exc(file=redirected_error)
        success = False
    finally:
        sys.stdout = old_stdout
        sys.stderr = old_stderr

    return {
        "success": success,
        "stdout": redirected_output.getvalue(),
        "stderr": redirected_error.getvalue()
    }

# ------------------------------------------------------------------------------
# 5. Launch Server & Tunnel
# ------------------------------------------------------------------------------
nest_asyncio.apply()
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

t = threading.Thread(target=run_server, daemon=True)
t.start()
time.sleep(2)

!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

tunnel = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://localhost:8000"], stderr=subprocess.PIPE, text=True)

while True:
    line = tunnel.stderr.readline()
    match = re.search(r'(https://[a-zA-Z0-9-]+\.trycloudflare\.com)', line)
    if match:
        print("\n" + "="*60)
        print("👉 ANTIGRAVITY CONTROLLER LIVE AT:", match.group(1))
        print("="*60 + "\n")
        break

In [ ]:
# %% Identity-verified dataset curation
!pip install -q insightface
import os, cv2, csv, hashlib, numpy as np
from collections import defaultdict
from insightface.app import FaceAnalysis

BASE = "/content/drive/MyDrive/MC Thùy Trang - testing/Ảnh chưa edit"
OUT = "/content/drive/MyDrive/MC Thùy Trang - testing/LoRA_Training_Dataset"
ANCHORS = [   # 5-10 REAL photos you verified yourself: one face each, varied angle/lighting
    "Event A/IMG_0001.jpg",
    "Event B/IMG_0102.jpg",
]
ACCEPT, REVIEW_LO = 0.50, 0.35   # cosine similarity; tune after seeing the anchor stats + review folder
MIN_FACE_PX, MIN_SHARP = 256, 60 # face height in original px; Laplacian variance on a 256px face crop
PER_FOLDER = 3                   # max kept per event folder (diversity)
for d in ("accepted", "review"):
    os.makedirs(f"{OUT}/{d}", exist_ok=True)

app = FaceAnalysis(name="buffalo_l", providers=["CUDAExecutionProvider", "CPUExecutionProvider"])
app.prepare(ctx_id=0, det_size=(1024, 1024))

def read(path, cap=2400):
    im = cv2.imread(path)
    if im is None:
        return None, None, 1.0
    s = min(1.0, cap / max(im.shape[:2]))
    return im, (cv2.resize(im, None, fx=s, fy=s, interpolation=cv2.INTER_AREA) if s < 1 else im), s

def sharp(small, b):
    x0, y0, x1, y1 = [max(0, int(v)) for v in b]
    g = cv2.cvtColor(small[y0:y1, x0:x1], cv2.COLOR_BGR2GRAY)
    return float(cv2.Laplacian(cv2.resize(g, (256, 256), interpolation=cv2.INTER_AREA), cv2.CV_64F).var())

def square_crop(im, b, s):
    H, W = im.shape[:2]
    x0, y0, x1, y1 = [v / s for v in b]
    fh, cx, cy = y1 - y0, (x0 + x1) / 2, (y0 + y1) / 2
    for k in ((5.5, 3.2) if fh / H < 0.15 else (3.2,)):      # half-body if face is small in frame
        side = int(k * fh)
        if side > min(H, W):
            continue
        cx0 = int(np.clip(cx - side / 2, 0, W - side)); cy0 = int(np.clip(cy - 1.3 * fh, 0, H - side))
        if cx0 <= x0 and cx0 + side >= x1 and cy0 <= y0 - 0.1 * fh and cy0 + side >= y1 + 0.3 * fh:
            c = im[cy0:cy0 + side, cx0:cx0 + side]
            return cv2.resize(c, (1024, 1024), interpolation=cv2.INTER_AREA) if side > 1024 else c
    return None

# --- anchor: centroid of hand-verified real photos
embs = []
for rel in ANCHORS:
    _, small, _ = read(os.path.join(BASE, rel))
    fs = app.get(small)
    assert len(fs) == 1, f"anchor needs exactly one face: {rel} ({len(fs)} found)"
    embs.append(fs[0].normed_embedding)
embs = np.array(embs)
pair = (embs @ embs.T)[np.triu_indices(len(embs), 1)]
print(f"anchor pairwise similarity: min {pair.min():.2f}, mean {pair.mean():.2f}  (ACCEPT should sit below these)")
anchor = embs.mean(0); anchor /= np.linalg.norm(anchor)

# --- scan
log, cands = [], defaultdict(list)
for root, _, files in os.walk(BASE):
    for f in sorted(files):
        if not f.lower().endswith((".jpg", ".jpeg", ".png")):
            continue
        p = os.path.join(root, f)
        im, small, s = read(p)
        if small is None:
            continue
        H, W = small.shape[:2]
        fs = app.get(small)
        if not fs:
            log.append((p, "no_face", "", "")); continue
        sims = [float(fc.normed_embedding @ anchor) for fc in fs]
        k = int(np.argmax(sims)); her, sim = fs[k], sims[k]
        b = her.bbox
        if sim < REVIEW_LO:
            st = "not_her"
        elif any(j != k and fs[j].det_score >= 0.5 and (fs[j].bbox[3] - fs[j].bbox[1]) >= 0.04 * H for j in range(len(fs))):
            st = "other_person"
        elif (b[3] - b[1]) / s < MIN_FACE_PX:
            st = "face_small"
        elif b[0] < 0.01 * W or b[1] < 0.01 * H or b[2] > 0.99 * W or b[3] > 0.99 * H:
            st = "cut_off"
        else:
            sh = sharp(small, b)
            if sh < MIN_SHARP:
                st = "blurry"
            else:
                cands[os.path.relpath(root, BASE)].append((sh, sim, p, list(b), s))
                log.append((p, "candidate", f"{sim:.3f}", f"{sh:.0f}")); continue
        log.append((p, st, f"{sim:.3f}", ""))

# --- per-folder cap (sharpest first), crop, split accepted / review
kept = {"accepted": 0, "review": 0, "no_crop": 0}
for folder, items in cands.items():
    for sh, sim, p, b, s in sorted(items, key=lambda t: -t[0])[:PER_FOLDER]:
        im = cv2.imread(p)
        c = square_crop(im, b, s)
        if c is None:
            kept["no_crop"] += 1; continue
        tag = "accepted" if sim >= ACCEPT else "review"
        name = f"{sim:.2f}__{folder[:20]}__{os.path.splitext(os.path.basename(p))[0]}__{hashlib.md5(p.encode()).hexdigest()[:6]}.jpg".replace(" ", "_").replace("/", "_")
        cv2.imwrite(f"{OUT}/{tag}/{name}", c, [cv2.IMWRITE_JPEG_QUALITY, 95])
        kept[tag] += 1

with open(f"{OUT}/curation_log.csv", "w", newline="", encoding="utf-8") as fh:
    csv.writer(fh).writerows([("path", "status", "sim", "sharp")] + log)
from collections import Counter
print(dict(Counter(r[1] for r in log)), kept)

In [ ]:
# %% CELL 1 - dataset prep (edit PHOTOS; paths are relative to BASE_PHOTOS)
import os
from PIL import Image, ImageOps

BASE_PHOTOS = "/content/drive/MyDrive/MC Thùy Trang - testing/Ảnh chưa edit"
DATASET = "/content/dataset"
PHOTOS = [
    "Event A/IMG_0001.jpg",
    "Event B/IMG_0102.jpg",
    # ... 15-25 sharp photos of her alone: varied angles, lighting, outfits; several close-ups
]
os.makedirs(DATASET, exist_ok=True)
for i, rel in enumerate(PHOTOS):
    im = ImageOps.exif_transpose(Image.open(os.path.join(BASE_PHOTOS, rel))).convert("RGB")
    s = 1024 / max(im.size)
    if s < 1:
        im = im.resize((round(im.width * s), round(im.height * s)), Image.LANCZOS)
    im.save(f"{DATASET}/{i:03d}.jpg", quality=95)
print(len(PHOTOS), "images ready")

# %% CELL 2 - install (Colab restarts may be needed once; re-run the cell after)
!pip install -q git+https://github.com/huggingface/diffusers peft accelerate bitsandbytes transformers sentencepiece
!wget -q -O /content/train_sdxl_lora.py https://raw.githubusercontent.com/huggingface/diffusers/main/examples/dreambooth/train_dreambooth_lora_sdxl.py
!accelerate config default

# %% CELL 3 - train (~1-2 h on a T4; re-run the same cell after a disconnect to resume)
OUT = "/content/drive/MyDrive/MC Thùy Trang - testing/lora_sdxl"
!accelerate launch /content/train_sdxl_lora.py \
  --pretrained_model_name_or_path=stabilityai/stable-diffusion-xl-base-1.0 \
  --pretrained_vae_model_name_or_path=madebyollin/sdxl-vae-fp16-fix \
  --instance_data_dir=/content/dataset \
  --instance_prompt="a photo of thuytrang_mc woman" \
  --output_dir="{OUT}" \
  --resolution=768 --train_batch_size=1 --gradient_accumulation_steps=1 \
  --gradient_checkpointing --use_8bit_adam --mixed_precision=fp16 \
  --rank=16 --learning_rate=1e-4 --lr_scheduler=constant --lr_warmup_steps=0 \
  --max_train_steps=1500 --checkpointing_steps=250 --checkpoints_total_limit=3 \
  --resume_from_checkpoint=latest --seed=42

# %% CELL 4 - generate (fp16 + CPU offload so it fits a T4)
import torch, glob
from diffusers import StableDiffusionXLPipeline, AutoencoderKL

vae = AutoencoderKL.from_pretrained("madebyollin/sdxl-vae-fp16-fix", torch_dtype=torch.float16)
pipe = StableDiffusionXLPipeline.from_pretrained(
    "SG161222/RealVisXL_V5.0", vae=vae, torch_dtype=torch.float16)   # photoreal fine-tune; check the repo name exists
# final weights, or point at e.g. OUT + "/checkpoint-1000" to compare earlier checkpoints
pipe.load_lora_weights(OUT, weight_name="pytorch_lora_weights.safetensors")
pipe.enable_model_cpu_offload()

img = pipe(
    "a photo of thuytrang_mc woman, hyperrealistic studio portrait, pastel blue oversized blazer, "
    "softbox lighting, rim light, 85mm lens, editorial fashion magazine",
        negative_prompt="cartoon, plastic skin, deformed, blurry, extra fingers",
    num_inference_steps=30, guidance_scale=5.0, height=1024, width=1024,
    cross_attention_kwargs={"scale": 0.9},   # LoRA strength: 1.0 = full likeness, lower if it looks overcooked
    generator=torch.Generator("cpu").manual_seed(1),
).images[0]
img.save("/content/portrait_test.png"); img

In [ ]:
from google.colab import drive
import shutil
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip uninstall -y opencv-python opencv-python-headless opencv-contrib-python
!pip install --no-cache-dir opencv-python-headless

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 MB 59.9 MB/s eta 0:00:00
